# Notebook B — SemSeg + transfer learning for environment classification

**Task:** same multi-label environment classification as Notebook A
(`forest, open_field, water, industry, city`), but via **semantic segmentation + transfer
learning**: fine-tune a pretrained SegFormer head on the 5 environment classes, then derive the
per-frame multi-label by thresholding each class's **pixel-area fraction**.

Outputs per-frame predictions to `dataset/eval/env_pred_semseg.csv` and runtime/frame to
`dataset/eval/runtime_semseg.json`, in the same format as Notebook A for `seg_evaluation.ipynb`.


## 0. Dependencies
Fine-tuning needs `datasets` + `accelerate` (run once if missing).

In [ ]:
# !pip install datasets accelerate
print("If the training imports fail, uncomment the pip line above and re-run.")

## 1. Setup

In [ ]:
import sys, json, time
from pathlib import Path

import numpy as np
import cv2
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
import segmentation_common as sc

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
ENV_CLASSES = sc.CATEGORIES["environment"]
ENV_ID = {c: i for i, c in enumerate(ENV_CLASSES)}   # env-only label space 0..4
print("Device:", DEVICE, "| environment classes:", ENV_CLASSES)
if DEVICE == "cpu":
    print("WARNING: training on CPU is slow - keep MAX_STEPS small for a smoke run.")

## 2. Configuration

In [ ]:
BASE_MODEL = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"
DATASET = "ADE20K"                       # "ADE20K" (runnable) | "MAPILLARY" (manual download)
MAPILLARY_ROOT = Path("../dataset/external/mapillary_vistas")
OUTPUT_DIR = Path("../models/segformer_env")

NUM_EPOCHS = 5
BATCH_SIZE = 4
LR = 6e-5
MAX_STEPS = None                         # set e.g. 50 for a quick smoke run
AREA_THRESHOLD = 0.03                    # class present if it covers >3% of the frame

TEST_IMAGES = Path("../dataset/eval/images")
PRED_CSV = Path("../dataset/eval/env_pred_semseg.csv")
RUNTIME_JSON = Path("../dataset/eval/runtime_semseg.json")

## 3. Label harmonization -> environment-only label space

Source class names are remapped to the 5 environment ids; everything else -> VOID (ignored).

In [ ]:
def ade20k_source_names():
    from huggingface_hub import hf_hub_download
    import json as _json
    p = hf_hub_download("huggingface/label-files", "ade20k-id2label.json", repo_type="dataset")
    id2label = _json.load(open(p))
    n = max(int(k) for k in id2label) + 1
    names = ["other"] * (n + 1)          # ADE masks are 1-indexed; 0 == other
    for k, v in id2label.items():
        names[int(k) + 1] = v
    return names


if DATASET == "ADE20K":
    source_names = ade20k_source_names()
    LUT = sc.build_id_lookup(source_names, sc.ADE20K_TO_TAXONOMY, class_id=ENV_ID)
elif DATASET == "MAPILLARY":
    import json as _json
    cfg = _json.load(open(MAPILLARY_ROOT / "config_v2.0.json"))
    source_names = [lbl["name"] for lbl in cfg["labels"]]
    LUT = sc.build_id_lookup(source_names, sc.MAPILLARY_TO_TAXONOMY, class_id=ENV_ID)
else:
    raise ValueError(DATASET)

print(f"{DATASET}: {int((LUT != sc.VOID_ID).sum())} source classes map into the 5 env classes")

## 4. Dataset & preprocessing

In [ ]:
from torch.utils.data import Dataset
from transformers import SegformerImageProcessor

processor = SegformerImageProcessor.from_pretrained(BASE_MODEL, do_reduce_labels=False)


class SegDataset(Dataset):
    def __init__(self, items):
        self.items = items                # list of (load_img, load_mask) callables

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        load_img, load_mask = self.items[i]
        mask = LUT[load_mask()].astype(np.uint8)             # remap to env ids
        enc = processor(load_img(), mask, return_tensors="pt")
        return {"pixel_values": enc["pixel_values"][0], "labels": enc["labels"][0]}


def build_ade20k_items(split):
    from datasets import load_dataset
    ds = load_dataset("scene_parse_150", split=split, trust_remote_code=True)
    return [(lambda ex=ds[i]: np.array(ex["image"].convert("RGB")),
             lambda ex=ds[i]: np.array(ex["annotation"])) for i in range(len(ds))]


if DATASET == "ADE20K":
    train_ds = SegDataset(build_ade20k_items("train"))
    val_ds = SegDataset(build_ade20k_items("validation"))
    print("train/val sizes:", len(train_ds), len(val_ds))

## 5. Model (transfer learning: new 5-class head)

In [ ]:
from transformers import SegformerForSemanticSegmentation

id2label = {i: c for c, i in ENV_ID.items()}
model = SegformerForSemanticSegmentation.from_pretrained(
    BASE_MODEL,
    num_labels=len(ENV_CLASSES),
    id2label=id2label,
    label2id=ENV_ID,
    ignore_mismatched_sizes=True,        # replace the Cityscapes head with a 5-class one
).to(DEVICE)
print("env loss ignore_index:", model.config.semantic_loss_ignore_index, "(== VOID 255)")

## 6. Train

In [ ]:
from transformers import TrainingArguments, Trainer


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = torch.tensor(logits)
    up = F.interpolate(logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
    preds = up.argmax(1).numpy()
    cm = np.zeros((len(ENV_CLASSES), len(ENV_CLASSES)), np.int64)
    for p, g in zip(preds, labels):
        valid = g != sc.VOID_ID
        idx = g[valid] * len(ENV_CLASSES) + p[valid]
        cm += np.bincount(idx, minlength=len(ENV_CLASSES) ** 2).reshape(cm.shape)
    tp = np.diag(cm); denom = cm.sum(0) + cm.sum(1) - tp
    iou = np.where(denom > 0, tp / np.clip(denom, 1, None), np.nan)
    return {"mIoU": float(np.nanmean(iou))}


args = TrainingArguments(
    output_dir=str(OUTPUT_DIR), learning_rate=LR, num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS if MAX_STEPS else -1,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="mIoU", greater_is_better=True, logging_steps=20,
    remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, compute_metrics=compute_metrics)
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
processor.save_pretrained(str(OUTPUT_DIR))
print("Saved fine-tuned env model to", OUTPUT_DIR)

## 7. SemSeg -> multi-label classifier (area threshold)

In [ ]:
@torch.no_grad()
def segment_env(image_rgb: np.ndarray) -> np.ndarray:
    enc = processor(image_rgb, return_tensors="pt").to(DEVICE)
    logits = model(**enc).logits
    up = F.interpolate(logits, size=image_rgb.shape[:2], mode="bilinear", align_corners=False)
    return up.argmax(1)[0].to(torch.uint8).cpu().numpy()


def classify_environment_semseg(image_rgb: np.ndarray, area_threshold: float = AREA_THRESHOLD) -> dict:
    """Multi-label prediction: a class is present if it covers > area_threshold of the frame."""
    mask = segment_env(image_rgb)
    n = mask.size
    return {c: int((mask == ENV_ID[c]).sum() / n > area_threshold) for c in ENV_CLASSES}

## 8. Predict over the test set + runtime

In [ ]:
def run_semseg_testset() -> pd.DataFrame:
    imgs = sorted(TEST_IMAGES.glob("*.jpg")) + sorted(TEST_IMAGES.glob("*.png"))
    if not imgs:
        print("No test images found - nothing to do.")
        return pd.DataFrame()

    rows, t0 = [], time.perf_counter()
    for fp in imgs:
        img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
        rows.append({"image": fp.name, **classify_environment_semseg(img)})
    ms = (time.perf_counter() - t0) / len(imgs) * 1000

    df = pd.DataFrame(rows)
    PRED_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(PRED_CSV, index=False)
    json.dump({"method": "semseg", "ms_per_frame": ms, "n": len(imgs)}, open(RUNTIME_JSON, "w"))
    print(f"Saved {len(df)} predictions -> {PRED_CSV}  |  {ms:.1f} ms/frame")
    return df


predictions = run_semseg_testset()
predictions.head()